# 实验一：卷积神经网络（CNN）花卉图像分类

本笔记本对应《深度学习实验任务书1：CNN 图像分类》，数据集为 Kaggle [Flower Recognition](https://www.kaggle.com/datasets/alxmamaev/flowers-recognition)（5 类花卉，约 4326 张图）。

**数据准备**：将解压后的类别文件夹重命名为 `0`、`1`、`2`、`3`、`4`（与任务书一致），并置于 `data/flowers/` 下，形如 `data/flowers/0/*.jpg`。

**说明**：任务书示例在 `forward` 末尾使用 `softmax` 且损失为 `CrossEntropyLoss`，二者在 PyTorch 中不应同时使用。本实验 **forward 输出 logits**，训练用 `CrossEntropyLoss`；若需概率可在推理时对 logits 做 `softmax`。


## 1. 依赖与环境

在仓库目录 `exp/exp1/` 下安装依赖（若需 GPU，请按 [PyTorch 官网](https://pytorch.org) 选择对应 CUDA 版本的安装命令）：

```bash
pip install -r requirements.txt
```



In [1]:
# 基础依赖
import os
import random
import zipfile
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import ImageFolder
from torchvision import transforms

# 固定随机种子便于复现
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


device: cpu


/home/hongchang/miniconda3/envs/exp/lib/python3.11/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## 2. 数据路径与预处理

与任务书一致：`Resize((28,28))`（加快实验）、`ToTensor`、`Normalize([0.5]*3, [0.5]*3)`。

若从 zip 解压，可将 zip 路径传给 `split_train_val_data` 的 `zip_path`；否则使用已解压目录 `data_dir`。


In [2]:
# 默认数据根目录（相对当前工作目录，一般在 exp/exp1 下打开笔记本）
DATA_DIR = os.environ.get("FLOWERS_DATA_DIR", "data/flowers")
ZIP_PATH = os.environ.get("FLOWERS_ZIP_PATH", "")  # 可选，例如 flowers.zip 的绝对路径

# 任务书示例变换
normalize = transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
transformer_image = transforms.Compose(
    [
        transforms.Resize((28, 28)),
        transforms.ToTensor(),
        normalize,
    ]
)


class FlowerDataset(Dataset):
    """任务书：继承 Dataset，实现 __len__ / __getitem__。"""

    def __init__(self, filenames, labels, transform):
        self.filenames = filenames
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        image = Image.open(self.filenames[idx]).convert("RGB")
        image = self.transform(image)
        return image, int(self.labels[idx])


def _find_imagefolder_root(base: Path) -> Path:
    """在解压目录下自动寻找可作为 ImageFolder 根的路径（兼容 Kaggle 多种目录结构）。"""
    candidates = [base]
    nested = base / "flowers"
    if nested.is_dir():
        candidates.append(nested)
    for p in sorted([x for x in base.iterdir() if x.is_dir()]):
        if p not in candidates:
            candidates.append(p)
    for c in candidates:
        try:
            ds = ImageFolder(str(c))
        except Exception:
            continue
        if len(ds.classes) >= 2:
            return c
    raise RuntimeError(
        f"无法在 {base} 下定位多类别 ImageFolder 根，请手动整理目录或直接使用 data_dir"
    )


def split_train_val_data(
    data_dir: str,
    ratio,
    batch_size: int = 20,
    transform=None,
    zip_path: Optional[str] = None,
    extract_to: Optional[str] = None,
    num_workers: int = 0,
):
    """按类别划分训练/验证集，返回 DataLoader。

    ratio: 如 [0.8, 0.2]，训练占比、验证占比之和应为 1。
    zip_path: 若提供，先解压到 extract_to（默认 data_dir 的父目录下 flowers_extract）。
    """
    if transform is None:
        transform = transformer_image

    root = Path(data_dir)
    if zip_path:
        zp = Path(zip_path)
        if not zp.is_file():
            raise FileNotFoundError(f"zip 不存在: {zp}")
        out_dir = Path(extract_to) if extract_to else root.parent / "flowers_extract"
        out_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zp, "r") as zf:
            zf.extractall(out_dir)
        dataset_root = _find_imagefolder_root(out_dir)
    else:
        dataset_root = root

    dataset = ImageFolder(str(dataset_root))
    n_classes = len(dataset.classes)
    character = [[] for _ in range(n_classes)]
    for path, y in dataset.samples:
        character[y].append(path)

    train_images, val_images = [], []
    train_labels, val_labels = [], []
    for i, paths in enumerate(character):
        n_train = int(len(paths) * ratio[0])
        for p in paths[:n_train]:
            train_images.append(p)
            train_labels.append(i)
        for p in paths[n_train:]:
            val_images.append(p)
            val_labels.append(i)

    train_ds = FlowerDataset(train_images, train_labels, transform)
    val_ds = FlowerDataset(val_images, val_labels, transform)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=max(1, len(val_ds)),
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    return train_loader, val_loader, dataset.class_to_idx, n_classes


## 3.  CNN 结构

两层卷积（ReLU + MaxPool）+ 三层全连接；**注意** `12 * 7 * 7` 对应输入 `28×28` 时的张量形状。输出维度为类别数，**不使用 softmax**（与 `CrossEntropyLoss` 配合）。


In [5]:
class CNN(nn.Module):
    """与任务书结构一致；forward 返回 logits。"""

    def __init__(self, in_channels: int = 3, n_classes: int = 5):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, 24, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(24, 12, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )
        self.fc1 = nn.Sequential(
            nn.Linear(12 * 7 * 7, 196),
            nn.ReLU(),
        )
        self.fc2 = nn.Sequential(
            nn.Linear(196, 84),
            nn.ReLU(),
        )
        self.fc3 = nn.Linear(84, n_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc3(x)
        return x


@torch.no_grad()
def accuracy_from_logits(logits, y):
    pred = logits.argmax(dim=1)
    return (pred == y).float().mean().item()


def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        bs = x.size(0)
        total_loss += loss.item() * bs
        total_acc += accuracy_from_logits(logits, y) * bs
        n += bs
    return total_loss / max(1, n), total_acc / max(1, n)


@torch.no_grad()
def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = loss_fn(logits, y)
        bs = x.size(0)
        total_loss += loss.item() * bs
        total_acc += accuracy_from_logits(logits, y) * bs
        n += bs
    return total_loss / max(1, n), total_acc / max(1, n)


## 4. 训练（基线模型）

超参数与任务书一致示例：`EPOCH=10`，`learning_rate=0.001`，Adam，`CrossEntropyLoss`。

**运行前**请确认 `DATA_DIR` 或环境变量 `FLOWERS_DATA_DIR` 指向正确；若有 zip，设置 `FLOWERS_ZIP_PATH` 或在下方代码中填写 `ZIP_PATH`。


In [ ]:
N_CLASSES = 5
BATCH_SIZE = 20
EPOCH = 10
learning_rate = 0.001
ratio = [0.8, 0.2]

# 按需修改
data_dir = DATA_DIR
zip_path = ZIP_PATH if ZIP_PATH else None

train_loader, val_loader, class_to_idx, n_classes = split_train_val_data(
    data_dir,
    ratio,
    batch_size=BATCH_SIZE,
    transform=transformer_image,
    zip_path=zip_path,
    num_workers=0,
)
print("class_to_idx:", class_to_idx, "n_classes:", n_classes)

model = CNN(in_channels=3, n_classes=n_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
loss_fn = nn.CrossEntropyLoss()

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
for epoch in range(EPOCH):
    tl, ta = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
    vl, va = evaluate(model, val_loader, loss_fn, device)
    history["train_loss"].append(tl)
    history["train_acc"].append(ta)
    history["val_loss"].append(vl)
    history["val_acc"].append(va)
    print(
        f"Epoch {epoch+1}/{EPOCH} | train loss: {tl:.4f} acc: {ta:.4f} | "
        f"val loss: {vl:.4f} acc: {va:.4f}"
    )

out_dir = Path("model")
out_dir.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "model_state": model.state_dict(),
        "class_to_idx": class_to_idx,
        "history": history,
    },
    out_dir / "cnn_baseline.pt",
)
print("已保存:", out_dir.resolve() / "cnn_baseline.pt")


class_to_idx: {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4} n_classes: 5
Epoch 1/10 | train loss: 1.3127 acc: 0.4093 | val loss: 1.1582 acc: 0.4763
Epoch 2/10 | train loss: 1.1440 acc: 0.5072 | val loss: 1.1710 acc: 0.5272
Epoch 3/10 | train loss: 1.0776 acc: 0.5550 | val loss: 1.0513 acc: 0.5838
Epoch 4/10 | train loss: 0.9746 acc: 0.6101 | val loss: 1.0012 acc: 0.6208
Epoch 5/10 | train loss: 0.9216 acc: 0.6333 | val loss: 0.9703 acc: 0.6104
Epoch 6/10 | train loss: 0.8554 acc: 0.6654 | val loss: 0.9895 acc: 0.6092


## 5. 训练曲线（损失与准确率）

用于报告「损失下降、验证准确率变化」及是否出现过拟合迹象。


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(history["train_loss"], label="train")
ax[0].plot(history["val_loss"], label="val")
ax[0].set_title("Loss")
ax[0].legend()
ax[1].plot(history["train_acc"], label="train")
ax[1].plot(history["val_acc"], label="val")
ax[1].set_title("Accuracy")
ax[1].legend()
plt.tight_layout()
plt.show()


## 拓展实验（1）：可视化卷积核与中间特征图

- 将第一层 `Conv2d` 的权重映射为灰度/彩色网格，观察学到的边缘/颜色模式。
- 对单张样本前向传播，截取 `conv1` 后部分通道做特征图展示。


In [ ]:
def show_conv1_kernels(model: CNN, max_k: int = 24):
    w = model.conv1[0].weight.detach().cpu()  # [out, in, k, k]
    w = w[:max_k]
    n = w.size(0)
    cols = 8
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.2, rows * 1.2))
    axes = np.array(axes).reshape(-1)
    for i in range(rows * cols):
        ax = axes[i]
        ax.axis("off")
        if i < n:
            img = w[i].permute(1, 2, 0).numpy()
            img = (img - img.min()) / (img.max() - img.min() + 1e-8)
            ax.imshow(np.clip(img, 0, 1))
    plt.suptitle("Conv1 kernels (normalized per filter)")
    plt.tight_layout()
    plt.show()


@torch.no_grad()
def show_feature_maps(model: CNN, image_path: str, n_show: int = 8):
    model.eval()
    img = Image.open(image_path).convert("RGB")
    x = transformer_image(img).unsqueeze(0).to(device)
    feat = model.conv1(x)
    f = feat[0, :n_show].cpu().numpy()
    cols = min(4, n_show)
    rows = int(np.ceil(n_show / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.2, rows * 2.2))
    axes = np.array(axes).reshape(-1)
    for i in range(rows * cols):
        ax = axes[i]
        ax.axis("off")
        if i < n_show:
            ax.imshow(f[i], cmap="viridis")
            ax.set_title(f"ch {i}")
    plt.suptitle("Feature maps after conv1 (sample image)")
    plt.tight_layout()
    plt.show()


# 使用训练集中第一张图作为示例（若数据不存在会报错，仅在有数据环境运行）
sample_paths = train_loader.dataset.filenames  # FlowerDataset
if len(sample_paths) > 0:
    show_conv1_kernels(model)
    show_feature_maps(model, sample_paths[0], n_show=8)
else:
    print("无样本路径，跳过可视化")


## 拓展实验（2）：加深网络结构并对比精度

在相近训练轮数下，增加一层卷积块并加入 `Dropout`，观察验证集准确率相对基线的变化（可在实验报告中用文字分析容量、过拟合与数据增强的取舍）。

以下为 **CNNDeep**：三层卷积 + 全连接；输入仍为 `28×28`，展平维度随池化自动计算。


In [ ]:
class CNNDeep(nn.Module):
    def __init__(self, in_channels: int = 3, n_classes: int = 5, p_drop: float = 0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        # 28 ->14->7->3
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 3 * 3, 256),
            nn.ReLU(),
            nn.Dropout(p_drop),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.fc(x)


deep_model = CNNDeep(in_channels=3, n_classes=n_classes).to(device)
opt2 = torch.optim.Adam(deep_model.parameters(), lr=learning_rate)
hist2 = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
for epoch in range(EPOCH):
    tl, ta = train_one_epoch(deep_model, train_loader, opt2, loss_fn, device)
    vl, va = evaluate(deep_model, val_loader, loss_fn, device)
    hist2["train_loss"].append(tl)
    hist2["train_acc"].append(ta)
    hist2["val_loss"].append(vl)
    hist2["val_acc"].append(va)
    print(
        f"[Deep] Epoch {epoch+1}/{EPOCH} | train loss: {tl:.4f} acc: {ta:.4f} | "
        f"val loss: {vl:.4f} acc: {va:.4f}"
    )

torch.save(
    {"model_state": deep_model.state_dict(), "class_to_idx": class_to_idx, "history": hist2},
    Path("model") / "cnn_deep.pt",
)

plt.figure(figsize=(6, 4))
plt.plot(history["val_acc"], label="baseline val acc")
plt.plot(hist2["val_acc"], label="deep val acc")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.legend()
plt.title("Validation accuracy: baseline vs deeper CNN")
plt.tight_layout()
plt.show()


## 保存实验数据与图片

在**完成上述训练与可视化单元格**之后运行本节：将指标写入 `experiment_outputs/cnn_flower/`，便于写报告或归档。若某变量不存在（例如未跑拓展），会跳过对应项。


In [ ]:
# 在基线训练完成后运行；若已执行拓展（2）且存在 hist2，会额外保存深度模型曲线与 deep_history.npz。
import json
from datetime import datetime

EXP_DIR = Path("experiment_outputs") / "cnn_flower"
EXP_DIR.mkdir(parents=True, exist_ok=True)

def _history_to_npz(prefix: str, h: dict) -> None:
    np.savez_compressed(
        EXP_DIR / f"{prefix}_history.npz",
        train_loss=np.asarray(h["train_loss"], dtype=np.float64),
        train_acc=np.asarray(h["train_acc"], dtype=np.float64),
        val_loss=np.asarray(h["val_loss"], dtype=np.float64),
        val_acc=np.asarray(h["val_acc"], dtype=np.float64),
    )

_history_to_npz("baseline", history)

meta = {
    "saved_at": datetime.now().isoformat(timespec="seconds"),
    "data_dir": str(data_dir),
    "zip_path": zip_path,
    "EPOCH": int(EPOCH),
    "BATCH_SIZE": int(BATCH_SIZE),
    "learning_rate": float(learning_rate),
    "ratio": ratio,
    "class_to_idx": class_to_idx,
    "n_classes": int(n_classes),
    "final_baseline_val_acc": float(history["val_acc"][-1]) if history["val_acc"] else None,
}
if "hist2" in globals():
    _history_to_npz("deep", hist2)
    meta["final_deep_val_acc"] = float(hist2["val_acc"][-1]) if hist2["val_acc"] else None

with open(EXP_DIR / "experiment_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

# 基线：损失与准确率曲线
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(history["train_loss"], label="train")
ax[0].plot(history["val_loss"], label="val")
ax[0].set_title("Loss")
ax[0].legend()
ax[1].plot(history["train_acc"], label="train")
ax[1].plot(history["val_acc"], label="val")
ax[1].set_title("Accuracy")
ax[1].legend()
plt.tight_layout()
fig.savefig(EXP_DIR / "fig_baseline_loss_acc.png", dpi=150, bbox_inches="tight")
plt.close(fig)

if "hist2" in globals():
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(history["val_acc"], label="baseline val acc")
    ax2.plot(hist2["val_acc"], label="deep val acc")
    ax2.set_xlabel("epoch")
    ax2.set_ylabel("accuracy")
    ax2.legend()
    ax2.set_title("Validation accuracy: baseline vs deeper CNN")
    fig2.tight_layout()
    fig2.savefig(EXP_DIR / "fig_baseline_vs_deep_val_acc.png", dpi=150, bbox_inches="tight")
    plt.close(fig2)

# 卷积核（基线 CNN 第一层）
w = model.conv1[0].weight.detach().cpu()
max_k = min(24, w.size(0))
w = w[:max_k]
n, cols = w.size(0), 8
rows = int(np.ceil(n / cols))
figk, axes = plt.subplots(rows, cols, figsize=(cols * 1.2, rows * 1.2))
axes = np.array(axes).reshape(-1)
for i in range(rows * cols):
    axes[i].axis("off")
    if i < n:
        img = w[i].permute(1, 2, 0).numpy()
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        axes[i].imshow(np.clip(img, 0, 1))
figk.suptitle("Conv1 kernels (normalized per filter)")
figk.tight_layout()
figk.savefig(EXP_DIR / "fig_conv1_kernels.png", dpi=150, bbox_inches="tight")
plt.close(figk)

# 特征图（训练集首张图）
sample_paths = getattr(train_loader.dataset, "filenames", [])
if sample_paths:
    model.eval()
    img = Image.open(sample_paths[0]).convert("RGB")
    x = transformer_image(img).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = model.conv1(x)
    n_show = min(8, feat.size(1))
    f = feat[0, :n_show].cpu().numpy()
    cols_fm = min(4, n_show)
    rows_fm = int(np.ceil(n_show / cols_fm))
    figf, axesf = plt.subplots(rows_fm, cols_fm, figsize=(cols_fm * 2.2, rows_fm * 2.2))
    axesf = np.array(axesf).reshape(-1)
    for i in range(rows_fm * cols_fm):
        axesf[i].axis("off")
        if i < n_show:
            axesf[i].imshow(f[i], cmap="viridis")
            axesf[i].set_title(f"ch {i}")
    figf.suptitle("Feature maps after conv1 (sample image)")
    figf.tight_layout()
    figf.savefig(EXP_DIR / "fig_conv1_feature_maps.png", dpi=150, bbox_inches="tight")
    plt.close(figf)

print("已保存到:", EXP_DIR.resolve())
print("  - baseline_history.npz, experiment_meta.json")
if "hist2" in globals():
    print("  - deep_history.npz")
print("  - fig_baseline_loss_acc.png, fig_conv1_kernels.png" + (", fig_conv1_feature_maps.png" if sample_paths else ""))
if "hist2" in globals():
    print("  - fig_baseline_vs_deep_val_acc.png")


## 实验报告撰写提示

1. **环境**：Python 版本、PyTorch/torchvision 版本、CPU/GPU。
2. **数据**：来源链接、类别映射 `0..4`、训练/验证划分比例。
3. **原理**：按任务书简述卷积、激活、池化、全连接与反向传播；说明为何使用 logits + `CrossEntropyLoss`。
4. **结果**：贴训练/验证损失与准确率曲线；给出最终验证准确率。
5. **拓展**：附卷积核/特征图截图；对比基线与加深网络的验证指标，分析参数量、是否过拟合、是否需数据增强（如随机裁剪、水平翻转等，可自行加在 `transforms.Compose` 中做第三次对比）。

**勿在本机运行**：将本仓库拷贝到实验机后，安装依赖、放入数据，再运行笔记本。
